# F1 Pit stop Prediction using Machine Learning

# Problem Statement

The Goal of this project is to predict whether a Formula 1 driver will pit on the next Lap using Race telemetry, Trye Degradation and Strategic race Features.

# This is a Binary Classification problem where:

->1 = Driver pits next lap

->0 = Driver does not pit next Lap

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV)
from sklearn.preprocessing import(LabelEncoder, StandardScaler, OneHotEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Install catboost
!pip install catboost -q
from catboost import CatBoostClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE
import shap

# Install optuna
!pip install optuna -q
import optuna

import joblib
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# Load Dataset

In [ ]:
# Loading Data

train_df = pd.read_csv("/train.csv")
test_df = pd.read_csv("/train.csv")

In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df["PitNextLap"].value_counts()

### Very Important Insight

In real Formula 1 Racs, Drivers do not pit every lap.pit stop events are relatively rare and occur only under specific startegic conditions such as tyre degradations, race strategy, traffic management, weather condistions, or safety Car events.

This creates a Class Imbalance problem in the dataset, where non-pit laps significantly outnumber pit-stop laps.

Handling This imbalance correctly is imortant for building a relible machine Learning Model.

# EXPLORATORY DATA ANALYSIS(EDA)

In [ ]:
plt.figure(figsize=(8,5))
sns.set_style("whitegrid")

sns.countplot(x=train_df["PitNextLap"], palette=["green", "red"])

plt.title("pit Next Lap Distribution")
plt.xlabel("pit Next Lap")
plt.ylabel("count")

plt.show()

### Observation

The dataset is highly imbalanced, with significantly more non-pit laps than pit-stop laps.

This reflects real Formula 1 race conditions, where pit stops occur relatively infrequently during a race.

Because of this imbalance, accuracy alone may not be a sufficient evaluation metric for the machine learning model.

Metrics such as:
- Precision
- Recall
- F1-Score
- ROC-AUC

will provide a more reliable evaluation of model performance.

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(data=train_df, x="PitNextLap", y="TyreLife", palette=["blue", "red"])

plt.title("Tyre Life vs Pit Stop Decision")
plt.xlabel("PitNextLap")
plt.ylabel("Tyre Life")

plt.show()

### Observation

The boxplot compares tyre life distributions between pit-stop and non-pit-stop laps.

Drivers who pit on the next lap generally show higher tyre life values, indicating increased tyre wear before pit stops.

This suggests that tyre degradation plays an important role in Formula 1 pit strategy decisions and may be a strong predictive feature for the machine learning model.

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(data=train_df, x="Compound", hue="PitNextLap")

plt.title("Tyre Compound vs Pit Decision")

plt.show()

### Observation

The distribution shows that pit-stop decisions vary across tyre compounds. Medium and hard tyres dominate the dataset, while intermediate and wet compounds appear much less frequently.

Hard tyres show a noticeable number of pit-next-lap cases, which may be linked to longer stint usage and eventual tyre degradation. Medium tyres are the most common compound overall, especially for non-pit laps.

This suggests that tyre compound contains useful strategic information and should be included as a categorical feature in the machine learning model.

In [ ]:
plt.figure(figsize=(12,6))

sns.histplot(data=train_df, x="LapNumber", hue="PitNextLap", bins=50, kde=True)

plt.title("Lap Number Distribution by Pit Decision")

plt.show()

### Observation

The histogram shows how pit-stop decisions are distributed across different race laps.

Pit-stop events tend to occur more frequently during specific sections of the race rather than being evenly distributed across all laps.

This reflects real Formula 1 race strategy, where teams typically plan pit stops around tyre degradation, fuel strategy, and race pace optimization.

The distribution suggests that LapNumber is likely an important predictive feature for identifying strategic pit windows.

In [ ]:
numeric_df = train_df.select_dtypes(include=np.number)

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")

plt.title("Correlation Heatmap")

plt.show()

### Removing Non-Predictive Identifier

The `id` column was removed because it functions only as a unique identifier and does not contain meaningful information related to race strategy or pit-stop behavior.

Keeping such identifiers may introduce noise into the machine learning model without improving predictive performance.

In [ ]:
train_df = train_df.drop("id", axis=1)

In [ ]:
train_df.columns

FEATURE ENGINEERING


In [ ]:
train_df["TyreLife_Ratio"] = ( train_df["TyreLife"] / train_df["LapNumber"])
train_df["PaceLoss_PerLap"] = (train_df["LapTime_Delta"] / (train_df["TyreLife"] + 1))
train_df["Deg_PerLap"] = (train_df["Cumulative_Degradation"] /(train_df["TyreLife"] + 1))
train_df["PitWindow"] = (((train_df["LapNumber"] >= 10) & (train_df["LapNumber"] <= 35)).astype(int))

In [ ]:
def race_phase(progress):

    if progress < 0.33:
        return "Early"

    elif progress < 0.66:
        return "Mid"

    else:
        return "Late"


train_df["RacePhase"] = (train_df["RaceProgress"].apply(race_phase))

### Race Phase Feature Engineering

The continuous `RaceProgress` variable was transformed into categorical race phases:
- Early Race
- Mid Race
- Late Race

This feature helps the model capture strategic differences in pit-stop behavior across different stages of the race.

In Formula 1, pit-stop strategies vary significantly depending on race phase, tyre degradation, fuel strategy, and race dynamics.

In [ ]:
train_df["Top10"] = ((train_df["Position"] <= 10).astype(int))
train_df["LosingPositions"] = ((train_df["Position_Change"] < 0).astype(int))
train_df["FreshTyres"] = ((train_df["TyreLife"] <= 3).astype(int))
train_df["LongStint"] = ((train_df["TyreLife"] >= 20).astype(int))
train_df["AggressiveDeg"] = ((train_df["LapTime_Delta"] >train_df["LapTime_Delta"].median()).astype(int))

In [ ]:
train_df.head()

# Engineered Features Added

The following additional features were created during feature engineering to help the model better capture Formula 1 race strategy and pit-stop behavior patterns.

| Feature Name | Description |
|---|---|
| TyreLife_Ratio | Ratio of tyre life relative to race progression |
| PaceLoss_PerLap | Pace degradation per tyre lap |
| Deg_PerLap | Tyre degradation intensity per lap |
| PitWindow | Indicates whether the lap falls within a common strategic pit window |
| RacePhase | Categorizes race into Early, Mid, or Late phase |
| Top10 | Indicates whether the driver is currently inside the top 10 |
| LosingPositions | Indicates whether the driver is losing race positions |
| FreshTyres | Indicates whether the tyres are newly fitted |
| LongStint | Indicates whether the driver has been on the same tyre set for a long time |
| AggressiveDeg | Indicates whether lap-time degradation is higher than average |

In [ ]:
train_df.shape

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.describe()

Splitting Data into Train and Test Data:

In [ ]:
X = train_df.drop("PitNextLap", axis=1)

y = train_df["PitNextLap"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
categorical_features = ["Driver", "Compound", "Race", "RacePhase"]

In [ ]:
numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

In [ ]:
print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

In [ ]:
numerical_transformer = Pipeline(steps=[("scaler", StandardScaler())])

In [ ]:
categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

# Combined Preprocessing Pipeline

A `ColumnTransformer` was used to apply different preprocessing techniques to numerical and categorical features.

- Numerical features were standardized using `StandardScaler`
- Categorical features were encoded using `OneHotEncoder`

This approach ensures that each feature type receives the appropriate preprocessing transformation before machine learning training.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("Processed Train Shape:", X_train_processed.shape)

print("Processed Test Shape:", X_test_processed.shape)

BASELINE MACHINE LEARNING MODELS

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression( max_iter=1000 )

log_model.fit( X_train_processed, y_train )

In [ ]:
y_pred_log = log_model.predict(X_test_processed)

y_prob_log = log_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
def evaluate_model( y_true, y_pred, y_prob, model_name):

    print(f"\n===== {model_name} =====")

    print("Accuracy:", accuracy_score(y_true, y_pred))

    print("Precision:", precision_score(y_true, y_pred))

    print("Recall:", recall_score(y_true, y_pred))

    print( "F1 Score:", f1_score(y_true, y_pred))

    print("ROC-AUC:", roc_auc_score(y_true, y_prob))

    print("\nClassification Report:\n")

    print(classification_report(y_true,y_pred))

In [ ]:
evaluate_model(
    y_test,
    y_pred_log,
    y_prob_log,
    "Logistic Regression"
)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier( n_estimators=100, max_depth=15, random_state=42,n_jobs=-1)

rf_model.fit(X_train_processed,y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test_processed)

y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
evaluate_model(y_test, y_pred_rf, y_prob_rf, "Random Forest")

In [ ]:
# XG BOOST
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_model.fit(
    X_train_processed,
    y_train
)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_processed)

y_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
evaluate_model( y_test, y_pred_xgb, y_prob_xgb, "XGBoost" )

# Model Comparison and Final Selection

Multiple machine learning models were trained and evaluated for predicting next-lap Formula 1 pit-stop decisions.

The models included:
- Logistic Regression
- Random Forest
- XGBoost

Each model was evaluated using:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

## Key Observations

- Logistic Regression provided a strong baseline model and demonstrated reasonable classification performance.
- Random Forest improved the model’s ability to capture non-linear relationships and strategic race patterns.
- XGBoost achieved the best overall performance across most evaluation metrics, particularly in ROC-AUC and recall.

## Final Model Selection

XGBoost was selected as the final production model because it demonstrated:
- strong predictive performance
- effective handling of high-dimensional encoded features
- improved detection of pit-stop events
- robustness on the imbalanced dataset

Its ability to model complex relationships between tyre degradation, race strategy, and telemetry features made it the most suitable model for this project.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

In [ ]:
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": xgb_model.feature_importances_})

In [ ]:
importance_df = importance_df.sort_values(by="Importance",ascending=False)

In [ ]:
plt.figure(figsize=(12,8))

top_features = importance_df.head(20)

sns.barplot(data=top_features,x="Importance",y="Feature")

plt.title("Top 20 Most Important Features")

plt.show()

In [ ]:
cm = confusion_matrix(y_test,y_pred_xgb)

plt.figure(figsize=(8,6))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")

plt.title("XGBoost Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

### Confusion Matrix Interpretation

The confusion matrix provides a detailed breakdown of the model’s classification performance.

It shows:
- correctly predicted pit stops
- correctly predicted non-pit laps
- false pit predictions
- missed pit-stop events

The XGBoost model demonstrates strong classification capability, particularly in identifying non-pit laps while also achieving good detection performance for actual pit-stop events.

The confusion matrix is especially important for imbalanced classification problems because it reveals the specific types of prediction errors made by the model.

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test,y_prob_xgb)

plt.figure(figsize=(8,6))

plt.plot(fpr,tpr,label=f"AUC = {roc_auc_score(y_test, y_prob_xgb):.3f}")

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve - XGBoost")

plt.legend()

plt.show()

### ROC Curve Interpretation

The ROC (Receiver Operating Characteristic) curve evaluates the model’s ability to distinguish between pit-stop and non-pit-stop events across different probability thresholds.

The XGBoost model achieved a very high ROC-AUC score, indicating excellent classification capability and strong separation between the two classes.

The curve remains significantly above the random baseline, demonstrating that the model successfully learned meaningful race strategy and tyre degradation patterns from the dataset.

ROC-AUC is especially important for this project because the dataset is imbalanced, making threshold-independent evaluation highly valuable.

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)

In [ ]:
shap_values = explainer.shap_values( X_test_processed)

In [ ]:
plt.figure(figsize=(12,8))

shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names)

In [ ]:
shap.summary_plot(shap_values,X_test_processed,feature_names=feature_names,plot_type="bar")

In [ ]:
def create_features(df):

    df = df.copy()

    df["TyreLife_Ratio"] = (df["TyreLife"] /df["LapNumber"])

    df["PaceLoss_PerLap"] = (df["LapTime_Delta"] /(df["TyreLife"] + 1))

    df["Deg_PerLap"] = (df["Cumulative_Degradation"] /(df["TyreLife"] + 1))

    df["PitWindow"] = (
        (
            (df["LapNumber"] >= 10) &
            (df["LapNumber"] <= 35)
        ).astype(int)
    )

    def race_phase(progress):

        if progress < 0.33:
            return "Early"

        elif progress < 0.66:
            return "Mid"

        else:
            return "Late"

    df["RacePhase"] = (
        df["RaceProgress"]
        .apply(race_phase)
    )

    df["Top10"] = (
        (df["Position"] <= 10)
        .astype(int)
    )

    df["LosingPositions"] = (
        (df["Position_Change"] < 0)
        .astype(int)
    )

    df["FreshTyres"] = (
        (df["TyreLife"] <= 3)
        .astype(int)
    )

    df["LongStint"] = (
        (df["TyreLife"] >= 20)
        .astype(int)
    )

    df["AggressiveDeg"] = (
        (
            df["LapTime_Delta"] >
            df["LapTime_Delta"].median()
        ).astype(int)
    )

    return df

In [ ]:
train_df = create_features(train_df)

In [ ]:
X = train_df.drop("PitNextLap", axis=1)

# Target variable
y = train_df["PitNextLap"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
categorical_features = ["Driver", "Compound", "Race", "RacePhase"]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_transformer = Pipeline(
    steps=[("scaler", StandardScaler())])

categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[("num", numerical_transformer, numerical_features),("cat", categorical_transformer, categorical_features)])

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape:", X_test_processed.shape)

In [ ]:
from xgboost import XGBClassifier

xgb_final_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_final_model.fit( X_train_processed, y_train)

In [ ]:
y_pred_final = xgb_final_model.predict(X_test_processed)

y_prob_final = xgb_final_model.predict_proba(X_test_processed)[:, 1]

evaluate_model(y_test, y_pred_final, y_prob_final, "Final XGBoost Model")

In [ ]:
from sklearn.pipeline import Pipeline
import joblib

final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_final_model)
    ]
)

joblib.dump(
    final_pipeline,
    "f1_pit_stop_final_pipeline.pkl"
)

print("Final production pipeline saved!")

In [ ]:
loaded_pipeline = joblib.load("f1_pit_stop_final_pipeline.pkl")

sample_data = X_test.head(5)

predictions = loaded_pipeline.predict(sample_data)

probabilities = loaded_pipeline.predict_proba(sample_data)[:, 1]

results = sample_data.copy()

results["Prediction"] = predictions

results["Pit_Probability"] = probabilities

results[
    [
        "Driver",
        "Compound",
        "LapNumber",
        "TyreLife",
        "Prediction",
        "Pit_Probability"
    ]
]